In [ ]:
#| default_exp vitest

# Vitest
> Run Vitest against an existing Chrome through fastcdp

`Vitest` owns the test runner and its hidden page, which has no visible tab or window. It connects to CDP Chrome on port 9223 unless you pass an existing `CDP` connection. It never launches or quits Chrome.

Install `vite-plus`, `@vitest/browser`, and `fastcdp-vitest` in your JavaScript project. Import `fastcdp` from `fastcdp-vitest` in `vite.config.js` and configure Browser Mode with `provider: fastcdp()` and one `{ browser: 'chromium' }` instance. Run `vp test` or `vp test watch` directly (the provider spawns `fastcdp-vitest-bridge` automatically), or drive runs from Python with `Vitest`.

Use ordinary `vitest-browser-svelte` rendering, Vitest locators and `expect.element`. Clicks, filling, typing, modifier shortcuts and Tab use native Chrome input. Holding non-modifier keys and unsupported action options raise errors. The provider does not install or launch a browser.

`run` accepts the same filename filters as `vp test`. It returns Vitest's native JSON report (`testResults`, `numPassedTests`, and so on) with `returncode` and combined `success`. Runner output is in `output`. Repeating `run` reuses the owned tab. The last component remains available for inspection until `close`.

In [ ]:
from fastcore.utils import *
from fastcore.script import call_parse
from fastcdp.core import CDP
from fastcdp.ext import ExtChannel
from tempfile import TemporaryDirectory
import asyncio, contextlib, json, secrets, signal, sys

In [ ]:
from fastcore.test import test_eq, expect_fail

## Running Vitest from Python

In [ ]:
class Vitest:
    "Run browser tests with fastcdp; retain the last test page until `close`."
    def __init__(self,
        root:str|Path='.', # JavaScript project containing the Vitest config and dependencies
        cdp:CDP=None, # Existing connection; otherwise connect to CDP Chrome on 9223
    ):
        self.root,self.cdp,self._owned = Path(root).resolve(),cdp,cdp is None
        self.page,self._proc = None,None

    async def __aenter__(self):
        if self.cdp is None: self.cdp = await CDP.remote()
        return self

    async def __aexit__(self, *exc): await self.close()

    async def close(self):
        "Close the owned test tab and connection, leaving Chrome running."
        if self.page: await self.page.close()
        if self._owned and self.cdp: await self.cdp.close()
    async def _commands(self, chan):
        while True:
            msg = await chan.events.get()
            try: await chan.send(id=msg.id, result=await self._action(msg))
            except Exception as e: await chan.send(id=msg.id, error=f'{type(e).__name__}: {e}')

`Vitest` connects to CDP Chrome on port 9223 on entry unless you supply an existing `CDP` connection. `close` always closes its hidden test tab, and closes the CDP connection only if `Vitest` created it; Chrome itself keeps running.

In [ ]:
@patch
async def run(self:Vitest,
    *filters:str, # Standard Vitest test-file filters
    env:dict=None, # Additional environment, e.g. the app's Rustygate URL
    timeout:float=60, # Seconds to wait for `vp test`; shutdown gets another five seconds
):
    "Return structured Vitest results, including failed assertions and nonzero exit status."
    if self._proc is not None: raise RuntimeError('A Vitest run is already active')
    if self.cdp is None: self.cdp = await CDP.remote()
    token = secrets.token_urlsafe()
    chan = await ExtChannel.listen(port=0, token=token)
    pump = asyncio.create_task(self._commands(chan))
    try:
        with TemporaryDirectory(prefix='fastcdp-vitest-') as tmp:
            report = Path(tmp)/'results.json'
            run_env = {**os.environ, **(env or {}), 'FASTCDP_VITEST': f'ws://127.0.0.1:{chan.port}/?token={token}'}
            self._proc = await asyncio.create_subprocess_exec('vp', 'test', '--reporter=default', '--reporter=json',
                f'--outputFile={report}', *filters, cwd=self.root, env=run_env,
                stdout=asyncio.subprocess.PIPE, stderr=asyncio.subprocess.STDOUT)
            comm = asyncio.create_task(self._proc.communicate())
            try:
                out,_ = await asyncio.wait_for(asyncio.shield(comm), timeout)
                self.output = out.decode()
                if not report.exists(): raise RuntimeError(self.output)
                data = report.read_json()
                data['returncode'] = self._proc.returncode
                data['success'] = bool(data.get('success')) and self._proc.returncode == 0
                return dict2obj(data)
            finally:
                if self._proc.returncode is None:
                    self._proc.terminate()
                    try: await asyncio.wait_for(self._proc.wait(), 5)
                    except TimeoutError:
                        self._proc.kill()
                        await self._proc.wait()
                out,_ = await comm
                self.output = out.decode()
    finally:
        self._proc = None
        pump.cancel()
        with contextlib.suppress(asyncio.CancelledError): await pump
        await chan.close()

`Vitest.run` opens a token-protected local `ExtChannel`, passes its WebSocket URL to `vp test` via `FASTCDP_VITEST`, and collects Vitest's JSON report from a temporary directory. If `vp test` exceeds `timeout`, `run` terminates the subprocess (falling back to `kill` after five seconds) and still captures its combined stdout and stderr in `self.output`.

Drive a project from Python with `async with Vitest(...)`:

In [ ]:
#| eval: false
#| exportd
from fastcdp.vitest import Vitest

async with Vitest('/path/to/svelte-project') as vt:
    result = await vt.run()
    result
    await vt.page.screenshot()

## Browser actions and input

In [ ]:
@patch
async def _node(self:Vitest, msg, target):
    p = self.page
    options = {k:v for k,v in (msg.get('options') or {}).items() if k in ('timeout','strict')}
    expr = f"""[...document.querySelectorAll('iframe[data-vitest]')]
        .find(f => new URL(f.src).searchParams.get('iframeId') === {json.dumps(msg.testPath)})
        .contentWindow.__fastcdp_find({json.dumps(target['selector'])}, {json.dumps(options)})"""
    r = await p.runtime.evaluate(expression=expr, awaitPromise=True)
    if 'exceptionDetails' in r:
        e = r['exceptionDetails']
        raise RuntimeError(e.get('exception', {}).get('description', e['text']))
    obj = r['objectId']
    try: return (await p.DOM.describeNode(objectId=obj))['backendNodeId']
    finally: await p.runtime.releaseObject(objectId=obj)

`_node` locates the Vitest test iframe matching `msg.testPath`, runs `__fastcdp_find` inside it with `timeout` and `strict` options, resolves the returned DOM node to a `backendNodeId`, and releases the remote object handle.

In [ ]:
@patch
async def _keyboard(self:Vitest, actions, unreleased=(), auto_close=False):
    modifiers = ('Alt','Control','Meta','Shift')
    pressed = set(unreleased)
    p = self.page
    def flags(): return dict(alt='Alt' in pressed, ctrl='Control' in pressed, meta='Meta' in pressed, shift='Shift' in pressed)
    async def modifier(key, down):
        if down: pressed.add(key)
        else: pressed.discard(key)
        await p.press(key, down=down, **flags())
    for a in actions:
        key = a['keyDef']['key']
        if key in ('\n','\r'): key = 'Enter'
        if key in modifiers:
            if key in pressed: await modifier(key, False)
            if not a['releasePrevious']:
                await modifier(key, True)
                if a['releaseSelf']: await modifier(key, False)
            continue
        if not a['releaseSelf'] or a['releasePrevious']: raise ValueError('Holding non-modifier keys is not supported')
        for _ in range(a['repeat']): await p.press(key, **flags())
    if auto_close:
        for key in tuple(pressed): await modifier(key, False)
    return dict(unreleased=list(pressed))

`_keyboard` translates Vitest user-event keyboard descriptors into native Chrome key events. It normalizes `\n` and `\r` to `Enter`, tracks held modifier keys (`Alt`, `Control`, `Meta`, `Shift`) across calls, and releases remaining modifiers when `auto_close=True`.

A small stub page lets us exercise modifier chords, newline normalization, and `auto_close` without launching Chrome:

In [ ]:
class _StubPage:
    def __init__(self): self.calls = []
    async def press(self, key, **kw): self.calls.append((key, kw))

def _act(key, *, release_self=True, release_prev=False, repeat=1):
    return dict(keyDef=dict(key=key), releaseSelf=release_self, releasePrevious=release_prev, repeat=repeat)

vt = Vitest()
vt.page = _StubPage()
res = await vt._keyboard([_act('Control', release_self=False), _act('a'), _act('\n')])
test_eq(res, dict(unreleased=['Control']))
test_eq(vt.page.calls, [
    ('Control', dict(down=True, alt=False, ctrl=True, meta=False, shift=False)),
    ('a', dict(alt=False, ctrl=True, meta=False, shift=False)),
    ('Enter', dict(alt=False, ctrl=True, meta=False, shift=False)),
])

Passing `auto_close=True` releases any modifiers still held from earlier actions, and attempting to hold a non-modifier key raises `ValueError`:

In [ ]:
vt.page.calls.clear()
test_eq(await vt._keyboard([], unreleased=res['unreleased'], auto_close=True), dict(unreleased=[]))
test_eq(vt.page.calls, [('Control', dict(down=False, alt=False, ctrl=False, meta=False, shift=False))])

with expect_fail(ValueError, contains='Holding non-modifier keys'): await vt._keyboard([_act('a', release_self=False)])

In [ ]:
@patch
async def _action(self:Vitest, m):
    if m.action == 'open':
        if self.page is None:
            target = await self.cdp.target.createTarget(url='about:blank', background=True, hidden=True)
            self.page = await self.cdp.attach_page(target)
        self.url = m.url
        await self.page.network.enable()
        await self.page.network.setCacheDisabled(cacheDisabled=True)
        await self.page.goto(m.url)
        return
    p = self.page
    if m.action == 'viewport':
        return await p.emulation.setDeviceMetricsOverride(width=m.width, height=m.height, deviceScaleFactor=1, mobile=False)
    if m.action == 'keyboard': return await self._keyboard(m.actions, m.unreleased)
    if m.action == 'cleanup': return await self._keyboard([], m.unreleased, auto_close=True)
    if m.action == 'tab': return await p.press('Tab', shift=m.get('shift', False))
    resolve = lambda: self._node(m, m.get('target') or dict(selector='body'))
    options = m.get('options') or {}
    if m.action == 'click': return await p.click(resolve, timeout=options.get('timeout', 5000)/1000)
    if m.action == 'hover': return await p.hover(resolve, timeout=options.get('timeout', 5000)/1000)
    if m.action not in ('fill', 'clear', 'type', 'screenshot'): raise ValueError(f'Unsupported fastcdp action: {m.action}')
    target = await resolve()
    if m.action in ('fill', 'clear'): return await p.fill_text(target, m.get('text', ''))
    if m.action == 'type':
        if not options.get('skipClick'): await p.DOM.focus(backendNodeId=target)
        return await self._keyboard(m.actions, options.get('unreleased', []), not options.get('skipAutoClose', False))
    box = (await p.DOM.getBoxModel(backendNodeId=target))['border']
    xs,ys = box[::2],box[1::2]
    return await p.page.captureScreenshot(clip=dict(x=min(xs), y=min(ys), width=max(xs)-min(xs), height=max(ys)-min(ys), scale=1),
        captureBeyondViewport=True)

`_action` handles each provider command: `open` creates one hidden background tab (with caching disabled) and navigates it to the Vitest URL, while `viewport`, `keyboard`, `cleanup`, `tab`, `click`, `hover`, `fill`, `clear`, `type`, and `screenshot` drive that page over CDP. Unknown actions raise `ValueError`:

In [ ]:
vt.page.calls.clear()
await vt._action(dict2obj(dict(action='tab', shift=True)))
test_eq(vt.page.calls, [('Tab', dict(shift=True))])

with expect_fail(ValueError, contains='Unsupported fastcdp action'): await vt._action(dict2obj(dict(action='unknown')))

## Direct CLI bridge

In [ ]:
async def _serve_bridge(port:int):
    stop = asyncio.Event()
    loop = asyncio.get_running_loop()
    for sig in (signal.SIGINT, signal.SIGTERM):
        with contextlib.suppress(NotImplementedError): loop.add_signal_handler(sig, stop.set)
    vt = Vitest()
    vt.cdp = await CDP.remote(port)
    async with vt:
        token = secrets.token_urlsafe()
        chan = await ExtChannel.listen(port=0, token=token)
        pump = asyncio.create_task(vt._commands(chan))
        fd = None
        def _on_stdin():
            nonlocal fd
            try:
                if os.read(fd, 1024): return
            except OSError: pass
            loop.remove_reader(fd)
            fd = None
            stop.set()
        if not sys.stdin.isatty():
            try:
                fd = sys.stdin.fileno()
                loop.add_reader(fd, _on_stdin)
            except (AttributeError, OSError, NotImplementedError): fd = None
        try:
            print(f'ws://127.0.0.1:{chan.port}/?token={token}', flush=True)
            await stop.wait()
        finally:
            if fd is not None: loop.remove_reader(fd)
            pump.cancel()
            with contextlib.suppress(asyncio.CancelledError): await pump
            await chan.close()

`_serve_bridge` binds a free local `ExtChannel`, prints its WebSocket URL to stdout, and stops cleanly on `SIGINT`, `SIGTERM`, or stdin EOF so the bridge never outlives a terminated parent Vitest process.

In [ ]:
@call_parse
def bridge_main(
    port:int=None, # Chrome CDP debug port (default: FASTCDP_PORT env or 9223)
):
    "Run the fastcdp WebSocket bridge for direct `vp test` runs and print its URL to stdout."
    asyncio.run(_serve_bridge(port or int(os.environ.get('FASTCDP_PORT', 9223))))

When you run `vp test` or `vp test watch` directly without Python driving the run, the `fastcdp-vitest` provider spawns `fastcdp-vitest-bridge` (`bridge_main`). It connects to CDP Chrome on `--port` (defaulting to `FASTCDP_PORT` or `9223`), opens an `ExtChannel`, prints its WebSocket URL to stdout, and exits on `SIGINT`, `SIGTERM`, or stdin EOF so it never outlives the parent Vitest process.